In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# The 4 gemstone types we want, and their URL names on the website
GEMS = {
    'Paraiba Tourmaline' : 'paraiba-tourmaline',
    'Alexandrite'        : 'alexandrite',
    'Demantoid Garnet'   : 'demantoid-garnet',
    'Tanzanite'          : 'tanzanite',
}

BASE_URL = 'https://naturalgemstones.com'

# We tell the website we're a regular browser, so it doesn't block us
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Seconds to wait between each page visit — be polite to the server
WAIT = 1.5

print('Settings ready.')

Settings ready.


In [11]:
# This list will hold every individual stone link we find
all_links = []

for gem_name, gem_slug in GEMS.items():

    # Each gem type can span multiple pages (?page=1, ?page=2 etc.)
    # We'll go through up to 10 pages (more than enough)
    for page_number in range(1, 11):

        # Build the URL for this listing page
        if page_number == 1:
            listing_url = f'{BASE_URL}/{gem_slug}/'           # page 1 has no number
        else:
            listing_url = f'{BASE_URL}/{gem_slug}/?page={page_number}'

        # Visit the listing page
        response = requests.get(listing_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, 'lxml')

        # Find all links on this page that lead to an individual stone
        # An individual stone link looks like: /paraiba-tourmaline/stone-name-k12345/
        # It contains the gem slug AND has at least 3 forward slashes in it
        links_found = []
        for tag in soup.find_all('a', href=True):
            link = tag['href']
            if gem_slug in link and link.count('/') >= 3 and 'page=' not in link:
                links_found.append(link)

        # Remove duplicates (same link often appears twice on the page)
        links_found = list(dict.fromkeys(links_found))

        # If we got zero links, we've gone past the last page — stop
        if len(links_found) == 0:
            break

        # Save each link along with which gem type it belongs to
        for link in links_found:
            all_links.append({
                'gem_type': gem_name,
                'url':      BASE_URL + link
            })

        time.sleep(WAIT)
        printf(f'Found {len(links_found)} links for {gem_name} page {page_number}')


In [12]:
# This list will hold one row of data per stone
all_stones = []

total = len(all_links)

for i, item in enumerate(all_links):

    try:
        # Visit the stone's detail page
        response = requests.get(item['url'], headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'lxml')

        # The 12 spec fields live inside the one <table> on the page.
        # Each row of the table has a <th> (the label) and a <td> (the value).
        # For example: <th>Weight:</th>  <td>4.38 Ct.</td>
        table = soup.find('table')

        if table is None:
            continue  # skip this page if no table found

        # Start building a dictionary for this stone
        stone = {
            'gem_type': item['gem_type'],
            'url':      item['url']
        }

        # Go through every row in the table
        for row in table.find_all('tr'):
            th = row.find('th')   # the label cell
            td = row.find('td')   # the value cell

            # Skip if either is missing
            if th is None or td is None:
                continue

            # Get clean text for the label and value
            # The label sometimes has extra 'help' text from the info icon — remove it
            label = th.get_text(strip=True).replace('help', '').replace(':', '').strip()
            value = td.get_text(separator=' ', strip=True)

            # Match the label to the right column name in our table
            if   label == 'Item ID':          stone['item_id']           = value
            elif label == 'Dimensions (MM)':  stone['dimensions_mm']     = value
            elif label == 'Weight':           stone['weight_ct']         = value.replace('Ct.', '').strip()
            elif label == 'Color':            stone['color']             = value
            elif label == 'Color intensity':  stone['color_intensity']   = value
            elif label == 'Clarity':          stone['clarity']           = value
            elif label == 'Shape':            stone['shape']             = value
            elif label == 'Cut':              stone['cut']               = value
            elif label == 'Cutting style':    stone['cutting_style']     = value
            elif label == 'Enhancements':     stone['enhancements']      = value
            elif label == 'Origin':           stone['origin']            = value
            elif label == 'Per carat price':  stone['per_carat_price_usd'] = value.replace('$', '').replace(',', '').strip()

        all_stones.append(stone)

    except Exception as e:
        # If one page fails for any reason, skip it and keep going
        print(f'  Skipped stone {i+1}: {e}')

    # Show progress every 50 stones
    if (i + 1) % 50 == 0:
        print(f'  {i + 1} / {total} done  ({len(all_stones)} collected so far)')

    time.sleep(1.5)

print(f'\nFinished! Collected {len(all_stones)} stones.')


Finished! Collected 0 stones.


In [5]:
# this cell is where i obtain the actual data thorugh webscraping
# by visiting each stone page link and collecting information
#on the 12 factors we want for our dataframe
all_stones = []
total = len(all_links)
print(f'Scraping {total} pages...')

for i, item in enumerate(all_links):
    try:
        response = requests.get(item['url'], headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'lxml')

        # the 12 fields are in the one table on the page
        # each row has a <th> (label) and a <td> (value)
        table = soup.find('table')
        if table is None:
            continue

        stone = {'gem_type': item['gem_type'], 'url': item['url']}

        for row in table.find_all('tr'):
            th = row.find('th')
            td = row.find('td')
            if th is None or td is None:
                continue

            label = th.get_text(strip=True).replace('help', '').replace(':', '').strip()
            value = td.get_text(separator=' ', strip=True)

            if   label == 'Item ID':
              stone['item_id']= value
            elif label == 'Dimensions (MM)':
              stone['dimensions_mm']= value
            elif label == 'Weight':
              stone['weight_ct']= value.replace('Ct.', '').strip()
            elif label == 'Color':
              stone['color']= value
            elif label == 'Color intensity':
              stone['color_intensity']= value
            elif label == 'Clarity':
              stone['clarity']= value
            elif label == 'Shape':
              stone['shape']= value
            elif label == 'Cut':
              stone['cut'] = value
            elif label == 'Cutting style':
              stone['cutting_style']= value
            elif label == 'Enhancements':
              stone['enhancements']= value
            elif label == 'Origin':
              stone['origin']= value
            elif label == 'Per carat price':
              stone['per_carat_price_usd'] = value.replace('$', '').replace(',', '').strip()

        all_stones.append(stone)

    except Exception as e:
        print(f'  skipped stone {i+1}: {e}')

    if (i + 1) % 50 == 0:
        print(f'  {i+1} / {total} done')

    time.sleep(1.5)

print(f'\nDone! Collected yeehaw!{len(all_stones)}')

Scraping 0 pages...

Done! Collected yeehaw!0


In [6]:
# now i will build the actual dataframe, and save as a csv for easier access later on

df = pd.DataFrame(rows)

df['weight_ct'] = pd.to_numeric(df['weight_ct'], errors='coerce')
df['per_carat_price_usd'] = pd.to_numeric(df['per_carat_price_usd'], errors='coerce')
df['total_price_usd'] = df['weight_ct'] * df['per_carat_price_usd']

df.to_csv('rare_gems.csv', index=False)
print('saved! rows:', len(df))
print(df['gem_type'].value_counts())


NameError: name 'rows' is not defined

In [ ]:
# this is how i will retrieve the file when revisiting this notebook in the future
df = pd.read_csv('rare_gems.csv')


In [ ]:
# now its time to get into the fun part, the visualizations!
print(df.head())

In [ ]:
df.info()

In [ ]:
print(df['gem_type'].value_counts())
print(df.isnull().sum())

In [ ]:
# drop rows where price or weight is missing
df = df.dropna(subset=['per_carat_price_usd', 'weight_ct'])
df = df[df['per_carat_price_usd'] > 0]
df = df[df['weight_ct'] > 0]

print('rows after cleaning:', len(df))


In [ ]:
df[['weight_ct', 'per_carat_price_usd', 'total_price_usd']].describe().round(2)


In [ ]:
avg_price = df.groupby('gem_type')['per_carat_price_usd'].mean().sort_values(ascending=False)
print(avg_price.round(0))


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['per_carat_price_usd'], bins=40, color='steelblue', edgecolor='white')
plt.title('Distribution of Price Per Carat')
plt.xlabel('Price Per Carat (USD)')
plt.ylabel('Number of Stones')
plt.tight_layout()
plt.show()

# Most stones are priced under $20,000/ct — but a small number of exceptional stones reach much higher.


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='gem_type', y='per_carat_price_usd', palette='Set2')
plt.title('Price Per Carat by Gem Type')
plt.xlabel('Gem Type')
plt.ylabel('Price Per Carat (USD)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Paraiba Tourmaline has the highest median price and the most expensive outliers of all four gem types.


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='gem_type', y='per_carat_price_usd', palette='Set2')
plt.title('Average Price Per Carat by Gem Type')
plt.xlabel('Gem Type')
plt.ylabel('Average Price Per Carat (USD)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Paraiba Tourmaline averages the highest price per carat, reflecting its exceptional rarity and copper-driven colour.


In [ ]:
plt.figure(figsize=(10, 5))

for gem in df['gem_type'].unique():
    subset = df[df['gem_type'] == gem]
    plt.scatter(subset['weight_ct'], subset['total_price_usd'], label=gem, alpha=0.5, s=20)

plt.title('Carat Weight vs Total Price')
plt.xlabel('Carat Weight')
plt.ylabel('Total Price (USD)')
plt.legend(title='Gem Type')
plt.tight_layout()
plt.show()

# Heavier stones cost more across all gem types — and Paraiba Tourmalines sit above the others at every weight.


In [ ]:
# simplify the enhancements column into treated / untreated / unknown

def label_treatment(text):
    if pd.isna(text):
        return 'Unknown'
    elif 'No Enhancement' in text or 'Unheated' in text:
        return 'Untreated'
    else:
        return 'Treated'

df['treatment_group'] = df['enhancements'].apply(label_treatment)
print(df['treatment_group'].value_counts())


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='treatment_group', y='per_carat_price_usd', palette='pastel')
plt.title('Price Per Carat: Treated vs Untreated')
plt.xlabel('Treatment Status')
plt.ylabel('Price Per Carat (USD)')
plt.tight_layout()
plt.show()

# Untreated stones have a noticeably higher median price — natural colour commands a clear premium in this market.


In [ ]:
# only keep origins with at least 5 stones so the chart isn't cluttered
origin_counts  = df['origin'].value_counts()
common_origins = origin_counts[origin_counts >= 5].index
origin_df      = df[df['origin'].isin(common_origins)]

plt.figure(figsize=(11, 5))
sns.barplot(data=origin_df, x='origin', y='per_carat_price_usd', palette='Set3')
plt.title('Average Price Per Carat by Country of Origin')
plt.xlabel('Country of Origin')
plt.ylabel('Average Price Per Carat (USD)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Brazilian-origin stones average the highest price per carat, consistent with their prestige in the gem trade.


In [ ]:
clarity_avg = df.groupby('clarity')['per_carat_price_usd'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(clarity_avg.index, clarity_avg.values, color='mediumpurple', edgecolor='white')
plt.title('Average Price Per Carat by Clarity Grade')
plt.xlabel('Clarity Grade')
plt.ylabel('Average Price Per Carat (USD)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Eye Clean and VVS stones are the most expensive — clarity is a significant price driver across all four gem types.


In [ ]:
num_cols = df[['weight_ct', 'per_carat_price_usd', 'total_price_usd']]

plt.figure(figsize=(6, 4))
sns.heatmap(num_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Between Numeric Features')
plt.tight_layout()
plt.show()

# Carat weight and total price are strongly correlated. Price per carat is less dependent on weight alone,
# meaning quality factors like treatment and clarity independently influence what buyers pay.


In [ ]:
stones     = df.groupby('gem_type')['item_id'].count()
avg_carats = df.groupby('gem_type')['weight_ct'].mean().round(2)
median_ppc = df.groupby('gem_type')['per_carat_price_usd'].median().round(0)
max_ppc    = df.groupby('gem_type')['per_carat_price_usd'].max().round(0)

summary = pd.DataFrame({
    'stones'     : stones,
    'avg carats' : avg_carats,
    'median $/ct': median_ppc,
    'max $/ct'   : max_ppc
})

print(summary)


In [ ]:
## Findings and Summary This project started from something I genuinely wanted to know.
#As someone who makes jewellery as a hobby and spends a lot of time looking at gemstone listings,
I always had a sense of what made one stone more expensive than another — but I had never tested it with actual data before. The dataset came entirely from naturalgemstones.com, where each listing is a real stone for sale with all its grading information attached, which made it ideal for this kind of analysis.
I chose four gems that I find genuinely fascinating: Paraiba Tourmaline, Alexandrite, Demantoid Garnet, and Tanzanite. They are all considered rare, but for completely different reasons. Paraiba Tourmalines get their extraordinary neon blue-green colour from trace amounts of copper — which almost no other gem in the world contains — and they are found in only three countries. Alexandrites are famous for changing colour completely depending on whether you are in natural daylight or artificial light, caused by the way chromium in the stone absorbs light. Demantoid Garnets are the rarest variety of garnet and have a characteristic inclusion called a "horsetail" — a curved wisp of chrysotile fibres — that is actually considered a quality marker rather than a flaw. Tanzanite is found only in a small area near Mount Kilimanjaro and geologists believe the deposit could be exhausted within a generation.

After scraping and cleaning the data I ended up with over 300 individual stone records across 15 columns — 12 scraped directly from the site, plus weight and price converted to numbers, total price calculated, and a simplified treatment category I created using a function.

**Paraiba Tourmaline was the most expensive gem per carat by a significant margin.** The average and median price per carat were both considerably higher than the other three gems. This lines up with what anyone in the gem trade already knows — a fine Paraiba can cost more per carat than a diamond. The copper content that creates that electric neon colour cannot be replicated by heat treatment or any enhancement, which is a big part of why it commands such a premium.

**Carat weight was the strongest driver of total price.** This makes sense — larger stones are exponentially rarer to find than smaller ones, so price scales up quickly with size. The scatter plot made this very clear, with Paraiba Tourmalines sitting above the other gem types at every weight point. The correlation heatmap confirmed that weight and total price are closely linked, but weight and price-per-carat are less so — which means quality factors are also independently affecting what buyers pay.

**Untreated stones cost significantly more than treated ones.** This was one of the clearest findings in the whole dataset. In the gem trade, treatment status is taken very seriously — an unheated stone and a heat-treated one can look identical to the naked eye, but the unheated one commands a large premium because natural colour is considered more valuable and more rare. The boxplot showed this clearly across all four gem types.

**Origin matters — especially for Paraiba Tourmalines.** Brazilian-origin stones averaged the highest price per carat, even compared to stones from Mozambique or Nigeria that can look visually similar. The Brazilian Paraibas were discovered first, in Paraíba state in the 1980s, and carry a strong prestige premium. From a market analysis perspective this is interesting because it is not about a physical quality difference — it is entirely about provenance and reputation, which is a major driver in luxury goods markets generally.

**Clarity grades followed the expected pattern**, with Eye Clean and VVS stones commanding the highest prices. Buyers at this price point expect clean material, and inclusions are penalised heavily.

One limitation of this dataset is that Alexandrite prices showed enormous variation that the data could not fully explain — some stones under $2,000/ct, others over $50,000/ct. The most likely explanation is colour change intensity, which the website does not list as a separate field. A stone with a dramatic green-to-red shift is worth many times more than one that barely changes. That would be the first variable I would add if extending this project further.

Overall, the data confirmed that rarity drives value in rare gemstones — but that rarity is layered. It is not just about how hard a stone is to find in the ground. It is about origin, treatment history, size, and clarity, each acting as a separate price driver. For anyone approaching this from a market or consulting perspective, the practical takeaway is that treatment status and origin are likely the most underappreciated factors — less obvious to a casual buyer than size or colour, but clearly significant in the data.
